# 00 数据准备
把Excel中的患者信息与配准影像按MR编号对应起来，生成后面分类实验共用的四层缓存。原图是512×512，缓存就保存512×512，不做空间缩放。

完成数据复核后，可将确认需要排除的整位患者文件夹移到 `patients_root` 之外保存。本次重建会按当前目录筛选患者清单；默认保留其余患者原来的结局、取层、开发/时间测试分组和五折编号。单个序列或标注文件缺失仍会报错，不自动排除患者。

在存放完整NIfTI的电脑上运行，完成后把新的 `cohort.csv` 和 `resnet50_manual_cache.npz` 一起传到算力机。


## 路径与临床字段
`config.json` 只记录数据位置、Excel 列名、结果目录和时间分界日期。MoCA 用于生成结局标签：≤22 记为 1；它不作为模型输入。年龄、受教育年限、NIHSS、同型半胱氨酸和尿酸用于临床对照及融合模型。


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "config.json").exists() and (ROOT / "1117" / "config.json").exists():
    ROOT = ROOT / "1117"
cfg = json.loads((ROOT / "config.json").read_text(encoding="utf-8"))

def path_from_config(key):
    value = Path(cfg[key]).expanduser()
    return value if value.is_absolute() else ROOT / value

CACHE = path_from_config("cache_path")
COHORT = path_from_config("cohort_path")
RESULTS = path_from_config("results_dir")
SEED = 20260922
CLINICAL = ["age", "education", "nihss", "hcy", "ua"]
SEQUENCES = ["t1", "t2", "dwi"]
IMAGE_FILES = ["t1_flair.nii.gz", "t2.nii.gz", "dwi_b1000.nii.gz"]
print("患者清单：", COHORT)
print("结果目录：", RESULTS)


患者清单： e:\git\PSCI\outputs\resnet_baseline_001\data\cohort.csv
结果目录： e:\git\PSCI\results


## 读取影像与人工标注
下面按 MR 编号寻找包含配准三序列的目录，读取 T1-FLAIR、T2、DWI 和 `Untitled.nii.gz`。

三序列和标注需要相同尺寸、相近空间矩阵。矩阵方向元素允许 0.0001、平移元素允许 0.001 的数值误差；超出时提示核对，不自动移动图像。这只是检查文件是否共用网格，实际配准效果还要看叠图。

`max_layer` 读取 MoCA 表的“脑室体素最大层”，沿用原代码：表中数值为 n 时，取第 n、n+1、n+2、n+3 层（层号从 1 开始）。范围外存在标注时提示后继续，取层位置仍由 Excel 决定。标签 1、2、3 分别保存为三个通道；其临床含义以人工标注规则为准。

`patients_root` 填直接包含各患者文件夹的目录；每例读取 `患者文件夹/配准图像`，患者文件夹名以 MR 编号结尾，例如 `姓名 MR250418095`。

目录扫描按整位患者的文件夹判断是否存在；即使T1文件缺失也不会把该患者当作已移出。`配准图像` 文件夹或其中任一必要文件缺失时，会在生成缓存前列出并报错。


In [2]:
import re
import nibabel as nib


def image_directories():
    root = path_from_config("patients_root")
    if not root.is_dir():
        raise FileNotFoundError(f"患者总目录不存在：{root}")

    folders = {}
    for patient_dir in sorted(root.iterdir()):
        if not patient_dir.is_dir():
            continue
        match = re.search(r"MR\s*(\d+)$", patient_dir.name.strip(), re.IGNORECASE)
        if not match:
            continue
        # 患者目录仍在就保留；缺少某个影像文件不能被当作整例已移出。
        directory = patient_dir / "配准图像"

        mr = "MR" + match.group(1)
        directory = directory.resolve()
        if mr in folders and folders[mr] != directory:
            raise ValueError(
                f"两个患者文件夹使用了同一个影像号 {mr}：\n"
                f"{folders[mr]}\n{directory}"
            )
        folders[mr] = directory

    if not folders:
        raise FileNotFoundError(
            f"未在 {root} 找到名称以MR编号结尾的患者文件夹。"
            "请把 patients_root 设为直接包含各患者文件夹的目录。"
        )
    return folders


def read_four_slices(patient, folders=None):
    recorded = [Path(str(patient.get(key, ""))) for key in SEQUENCES + ["manual_mask"]]
    if not all(file.is_file() for file in recorded):
        folders = folders if folders is not None else image_directories()
        if patient.mr not in folders:
            raise FileNotFoundError(f"找不到{patient.mr}的三序列目录。")
        recorded = [folders[patient.mr] / name for name in IMAGE_FILES + ["Untitled.nii.gz"]]
    volumes = [nib.load(str(file)) for file in recorded]
    ref = volumes[0]
    tolerance = np.array([1e-4, 1e-4, 1e-4, 1e-3])
    if len(ref.shape) != 3 or any(v.shape != ref.shape or not np.allclose(v.affine, ref.affine, atol=tolerance, rtol=0) for v in volumes[1:]):
        raise ValueError(f"{patient.mr}三序列与标注的shape/affine不一致。")
    layer = float(patient.max_layer)
    if not np.isfinite(layer) or not layer.is_integer():
        raise ValueError(f"{patient.mr}起始层不是整数。")
    start = int(layer) - 1
    if start < 0 or start + 4 > ref.shape[2]:
        raise ValueError(f"{patient.mr}四层取层越界。")
    labels = np.asarray(volumes[-1].dataobj)
    if not np.isfinite(labels).all() or not set(np.unique(labels)) <= {0, 1, 2, 3}:
        raise ValueError(f"{patient.mr}标签必须为0/1/2/3。")
    if np.any(labels[:, :, :start]) or np.any(labels[:, :, start + 4:]):
        print(f"提示：{patient.mr}四层之外仍有标注；本次按Excel取第{start + 1}～{start + 4}层，其余层不参与计算。")
    images = [np.asarray(v.dataobj, dtype=np.float32)[:, :, start:start + 4] for v in volumes[:3]]
    if any(not np.isfinite(image).all() for image in images):
        raise ValueError(f"{patient.mr}影像包含无效数值。")
    return images, labels[:, :, start:start + 4].astype(np.uint8)


def normalize_four(volume):
    lo, hi = np.percentile(volume, [1, 99.5])
    return np.clip((volume - lo) / max(float(hi - lo), 1e-6), 0, 1)


e:\anaconda\env\pytorch\Lib\site-packages\nibabel\optpkg.py:104: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.5)
  pkg = __import__(name, fromlist=fromlist)


## 匹配患者、划分队列、生成缓存
每个序列在选定四层内做1%～99.5%分位归一化，得到0～1的强度值。影像和人工标注均直接保存原始像素网格，不做缩放或插值。标签1、2、3分别转成三个0/1通道。

缓存形状为 `[患者数, 4, 3, 高, 宽]`。同一批患者需要相同的矩阵尺寸；如遇不同尺寸，这里会指出具体患者，停止写出缓存，便于检查数据。尺寸相同不代表配准一定正确，仍需查看叠图。

首次生成患者清单时，检查日期不晚于配置分界日的患者进入开发队列，之后进入时间测试队列；开发队列生成固定的患者级五折。此次重建缓存沿用已有清单和折号。临床缺失值保留到训练折内再填补。


In [3]:
from sklearn.model_selection import StratifiedKFold


def normalize_mr(value):
    text = str(value).strip().upper()
    match = re.fullmatch(r"(?:MR\s*)?(\d+)(?:\.0+)?", text)
    if not match:
        raise ValueError(f"无法识别影像号：{value}")
    return "MR" + match.group(1)


def make_cohort():
    source = pd.read_excel(path_from_config("excel_path"), sheet_name=cfg["sheet_name"])
    source.columns = source.columns.astype(str).str.strip()
    columns = cfg["columns"]
    required = ["mr", "moca", "max_layer", "check_time"] + CLINICAL
    missing = [columns[key] for key in required if columns[key] not in source.columns]
    if missing or not source.columns.is_unique:
        raise ValueError(f"请核对Excel列名；缺少：{missing}")
    cohort = pd.DataFrame({key: source[columns[key]] for key in required})
    cohort["moca"] = pd.to_numeric(cohort.moca, errors="coerce")
    usable = cohort.moca.between(0, 30)
    if not usable.all():
        print(f"排除{int((~usable).sum())}例缺失或无效MoCA。")
    cohort = cohort.loc[usable].copy()
    cohort["mr"] = cohort.mr.map(normalize_mr)
    if not cohort.mr.is_unique:
        raise ValueError("Excel中MR编号重复，请先确认每例患者使用哪次检查。")
    cohort["check_time"] = pd.to_datetime(cohort.check_time.astype(str).str.strip().str.replace(".", "-", regex=False),
                                           format="mixed", errors="coerce").dt.normalize()
    cohort["max_layer"] = pd.to_numeric(cohort.max_layer, errors="coerce")
    bad = cohort.check_time.isna() | ~np.isfinite(cohort.max_layer) | (cohort.max_layer % 1 != 0) | (cohort.max_layer < 1)
    if bad.any():
        raise ValueError(f"以下患者的检查日期或起始层有误：{cohort.loc[bad, 'mr'].tolist()}")
    cohort["max_layer"] = cohort.max_layer.astype(int)
    for key in CLINICAL:
        cohort[key] = pd.to_numeric(cohort[key], errors="coerce").replace([np.inf, -np.inf], np.nan)
    cohort["label"] = (cohort.moca <= 22).astype(int)
    cohort = cohort.sort_values(["check_time", "mr"], kind="mergesort").reset_index(drop=True)
    cohort["split"] = np.where(cohort.check_time <= pd.Timestamp(cfg["test_split_date"]), "development", "validation")
    cohort["development_fold"] = 0
    dev_index = np.flatnonzero(cohort.split.eq("development"))
    y = cohort.label.to_numpy()[dev_index]
    if np.bincount(y, minlength=2).min() < 5:
        raise ValueError("开发队列每类至少需要5例才能生成五折。")
    for fold, (_, valid) in enumerate(StratifiedKFold(5, shuffle=True, random_state=SEED).split(y, y), 1):
        cohort.loc[dev_index[valid], "development_fold"] = fold
    folders = image_directories()
    # Excel可保留已经移出的患者记录；下方统一按现有患者目录筛选。
    for key, filename in zip(SEQUENCES + ["manual_mask"], IMAGE_FILES + ["Untitled.nii.gz"]):
        cohort[key] = [str(folders[mr] / filename) if mr in folders else "" for mr in cohort.mr]
    cohort["cache_index"] = np.arange(len(cohort))
    return cohort


def retain_present_patients(cohort):
    folders = image_directories()
    present = cohort.mr.isin(folders)
    excluded = cohort.loc[~present, ["mr", "split", "development_fold"]].copy()
    excluded["reason"] = "当前patients_root中没有此患者目录；请核对是否为人工移出"
    kept = cohort.loc[present].copy().reset_index(drop=True)
    if kept.empty:
        raise ValueError("现有目录与患者清单没有交集，未写出缓存。请先核对patients_root。")
    print("当前患者总目录：", path_from_config("patients_root"))
    print(f"清单原有{len(cohort)}人，保留{len(kept)}人，目录不在的{len(excluded)}人。")
    if not excluded.empty:
        display(excluded)
        print("以上应与人工移出的名单一致；若不是，请停止本单元并核对路径。", flush=True)

    # 只过滤已不在患者总目录中的整例；现存患者缺文件会明确报错。
    missing_files = []
    for key, filename in zip(SEQUENCES + ["manual_mask"], IMAGE_FILES + ["Untitled.nii.gz"]):
        files = [folders[mr] / filename for mr in kept.mr]
        missing_files.extend(f"{mr}: {file}" for mr, file in zip(kept.mr, files) if not file.is_file())
        # 换电脑或移动数据后使用当前目录，避免继续读取清单里的旧路径。
        kept[key] = list(map(str, files))
    if missing_files:
        raise FileNotFoundError("以下患者目录仍在但文件不完整，本次未将这些患者自动排除：\n" + "\n".join(missing_files))

    development = kept.loc[kept.split == "development"]
    if set(development.development_fold) != {1, 2, 3, 4, 5}:
        raise ValueError("筛选后开发集缺少某一折，未写出缓存；请核对移出范围。")
    bad_folds = [int(fold) for fold, group in development.groupby("development_fold") if set(group.label) != {0, 1}]
    if bad_folds:
        raise ValueError(f"筛选后第{bad_folds}折缺少阳性或阴性，无法计算AUC；未自动重分组。")
    kept["cache_index"] = np.arange(len(kept))
    return kept, excluded


def build_classification_cache(cohort):
    images, masks, spatial_shape = None, None, None
    inspection = []
    for i, (_, patient) in enumerate(cohort.iterrows()):
        volumes, labels = read_four_slices(patient)
        h, w = labels.shape[:2]
        if spatial_shape is None:
            spatial_shape = (h, w)
            images = np.empty((len(cohort), 4, 3, h, w), dtype=np.float16)
            masks = np.empty(images.shape, dtype=np.uint8)
            print(f"原尺寸缓存：每层{h}×{w}，影像和mask均不缩放", flush=True)
        elif (h, w) != spatial_shape:
            raise ValueError(f"{patient.mr}原图为{h}×{w}，此前患者为{spatial_shape}。请核对配准影像尺寸；本次不会自动缩放，缓存尚未写出。")
        normalized = [normalize_four(v) for v in volumes]
        for s in range(4):
            for channel, volume in enumerate(normalized):
                images[i, s, channel] = volume[:, :, s]
            for label in (1, 2, 3):
                masks[i, s, label - 1] = (labels[:, :, s] == label).astype(np.uint8)
        inspection.append({"mr": patient.mr, "first_slice": int(patient.max_layer), "last_slice": int(patient.max_layer) + 3,
                           "height": h, "width": w,
                           **{f"label_{label}_pixels": int((labels == label).sum()) for label in (1, 2, 3)}})
        print(f"四层缓存 {i + 1}/{len(cohort)}：{patient.mr}", flush=True)
    if spatial_shape is None:
        raise ValueError("患者清单为空，未生成缓存。")
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    COHORT.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(CACHE, images=images, masks=masks, labels=cohort.label.to_numpy(dtype=np.int64),
                        mrs=cohort.mr.to_numpy(dtype=str), max_layers=cohort.max_layer.to_numpy(dtype=np.int16),
                        spatial_shape=np.asarray(spatial_shape, dtype=np.int32))
    cohort.to_csv(COHORT, index=False)
    pd.DataFrame(inspection).to_csv(COHORT.parent / "image_inspection.csv", index=False)
    return cohort


In [4]:
def read_cases():
    cases = pd.read_csv(COHORT, dtype={"mr": str})
    if not cases.mr.is_unique or not set(cases.split) <= {"development", "validation"}:
        raise ValueError("患者编号重复或分组有误，请检查cohort.csv。")
    if not set(cases.label) <= {0, 1} or cases.mr.isna().any():
        raise ValueError("患者编号缺失或结局不是0/1。")
    return cases


def read_classification_data():
    cases = read_cases()
    with np.load(CACHE, allow_pickle=False) as data:
        cache_mrs = data["mrs"].astype(str).tolist()
        if len(set(cache_mrs)) != len(cache_mrs) or set(cache_mrs) != set(cases.mr):
            raise ValueError("缓存与患者清单的MR编号不一致。")
        lookup = {mr: i for i, mr in enumerate(cache_mrs)}
        order = [lookup[mr] for mr in cases.mr]
        if not np.array_equal(data["labels"][order], cases.label) or not np.array_equal(data["max_layers"][order], cases.max_layer):
            raise ValueError("患者标签或取层与缓存不一致。")
        images = data["images"][order].astype(np.float32)
        masks = data["masks"][order].astype(np.uint8)
    if images.ndim != 5 or images.shape[1:3] != (4, 3) or masks.shape != images.shape or min(images.shape[-2:]) < 32:
        raise ValueError("分类输入应为[N,4,3,H,W]，影像与mask尺寸须一致，H和W至少为32。")
    if not np.isfinite(images).all() or not np.isin(masks, [0, 1]).all():
        raise ValueError("影像存在无效值或mask不是0/1。")
    return cases, images, masks


## 生成或查看数据
本次保持 `REBUILD_DATA=True`、`UPDATE_COHORT=False`。00先读取已有清单，再按 `patients_root` 当前实际存在的患者目录筛选，显示保留人数及目录不在的MR名单。请先确认路径正确、显示的名单确实是人工移出的患者，不能把算力机上的不完整数据副本当作完整患者目录运行。

`UPDATE_COHORT=False` 保留其余患者的原分组和折号，仅重建当前患者的原尺寸缓存。首次没有清单时才从Excel生成；只有确认需要更新Excel内容或新增患者时才设为 `True`，这时会重新生成清单和折号。移走患者后不需要在Excel删行。

现存患者缺少文件、空间网格异常或影像不可读时，仍停止并提示，不因为这些异常自动删除病例。若移出后某折缺少一类患者，也会提示，不自动重新分组。

重建成功后会覆盖配置指定的缓存和清单，并将本次移出MR追加到 `excluded_patients.csv`。该CSV仅记录目录筛选历史，实际医学排除原因请继续写入07的 `review_notes.csv`，不新增配置文件。重新放回一个此前已排除的患者目录，不会自动把他加回已覆盖的清单，需要另行核对并恢复其原记录。

完成后检查实际尺寸和各组人数，将新清单与缓存一起上传算力机。之后只查看缓存时设 `REBUILD_DATA=False`。无需删除整个outputs目录；`image_inspection.csv` 仍记录实际纳入患者的四层范围、尺寸及标签像素数。


In [ ]:
REBUILD_DATA = True  # 从原始NIfTI重建原尺寸缓存
UPDATE_COHORT = False  # 本次沿用现有患者及折号；只有更新Excel或新增患者时才设True

if REBUILD_DATA:
    cases = make_cohort() if UPDATE_COHORT or not COHORT.exists() else read_cases()
    cases, excluded = retain_present_patients(cases)
    display(cases.groupby("split").agg(n=("mr", "size"), positive=("label", "sum")))
    build_classification_cache(cases)
    if not excluded.empty:
        # 重建成功后追加排除记录；这是历史记录，不作为下次训练的排除配置。
        exclusion_log = COHORT.parent / "excluded_patients.csv"
        excluded["recorded_at"] = pd.Timestamp.now().isoformat(timespec="seconds")
        excluded.to_csv(exclusion_log, mode="a", header=not exclusion_log.exists(), index=False, encoding="utf-8")
        print("本次目录排除记录已追加：", exclusion_log)
elif not COHORT.exists() or not CACHE.exists():
    print("尚未找到缓存。确认config.json中的数据路径后，将REBUILD_DATA改为True。")

if COHORT.exists() and CACHE.exists():
    cases, images, masks = read_classification_data()
    print("缓存中每层实际尺寸：", images.shape[-2:])
    display(cases.groupby("split").agg(n=("mr", "size"), positive=("label", "sum")))
    display(cases[["mr", "moca", "label", "max_layer", "split", "development_fold"]].head())


## 看一例缓存
三行灰度图对应三种序列，最后一行把三个标签显示为红、绿、蓝。检查四层位置以及病灶与影像是否对得上。`EXAMPLE_MR` 留空时显示第一例。


In [ ]:
EXAMPLE_MR = ""

if COHORT.exists() and CACHE.exists():
    import matplotlib.pyplot as plt
    i = cases.index[cases.mr == EXAMPLE_MR][0] if EXAMPLE_MR else 0
    fig, axes = plt.subplots(4, 4, figsize=(10, 10))
    for s in range(4):
        for c, name in enumerate(SEQUENCES):
            axes[c, s].imshow(images[i, s, c], cmap="gray", vmin=0, vmax=1)
            axes[c, s].set_title(f"{name} | slice {int(cases.iloc[i].max_layer) + s}")
        axes[3, s].imshow(images[i, s, 0], cmap="gray", vmin=0, vmax=1)
        color = masks[i, s].transpose(1, 2, 0).astype(float)
        axes[3, s].imshow(np.dstack([color, color.max(axis=-1) * .6]))
    for ax in axes.flat:
        ax.axis("off")
    fig.suptitle(cases.iloc[i].mr)
    fig.tight_layout()
    plt.show()
